# xView2 EDA

This notebook summarizes the `hold`, `tier1`, and `tier3` splits.

It reports:
- scene-level damage class distribution
- building-level damage class distribution
- natural disaster type distribution
- disaster event distribution

Notes:
- Scene-level damage class uses the maximum building damage in the post-disaster label, which matches the tile-level classification target used in the current training code.
- Building-level damage class counts each building annotation by its subtype.
- Disaster type is extracted from label metadata when available, and otherwise inferred from the disaster event name.


In [ ]:
from pathlib import Path
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

DATA_ROOT = Path("/sfs/gpfs/tardis/home/xxn2pb/siamese_project/data/xview2_jpeg")
SPLITS = ["hold", "tier1", "tier3"]
OUTPUT_DIR = Path("eda_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

DAMAGE_NAME_TO_ID = {
    "background": 0,
    "no-damage": 1,
    "minor-damage": 2,
    "major-damage": 3,
    "destroyed": 4,
}

DAMAGE_ID_TO_NAME = {
    0: "background",
    1: "no-damage",
    2: "minor-damage",
    3: "major-damage",
    4: "destroyed",
}

DAMAGE_ORDER = [
    "background",
    "no-damage",
    "minor-damage",
    "major-damage",
    "destroyed",
]

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

print(f"DATA_ROOT = {DATA_ROOT}")
for split in SPLITS:
    print(f"{split}: {(DATA_ROOT / split).exists()}")


In [ ]:
def normalize_label_name(name):
    return str(name).strip().lower().replace("_", "-")


def normalize_disaster_type(raw_value):
    raw_value = normalize_label_name(raw_value)
    if not raw_value or raw_value == "none":
        return "unknown"

    aliases = {
        "flood": "flood",
        "flooding": "flood",
        "earthquake": "earthquake",
        "hurricane": "hurricane",
        "wind": "wind",
        "wind-damage": "wind",
        "fire": "fire",
        "wildfire": "fire",
        "tsunami": "tsunami",
        "volcano": "volcano",
        "volcanic-eruption": "volcano",
    }
    return aliases.get(raw_value, raw_value)


def extract_disaster_fields(label_data):
    metadata = label_data.get("metadata", {}) or {}

    disaster_event = (
        metadata.get("disaster")
        or metadata.get("disaster_name")
        or metadata.get("event_name")
        or label_data.get("disaster")
        or label_data.get("disaster_name")
        or label_data.get("event_name")
        or "unknown"
    )
    disaster_event = normalize_label_name(disaster_event)
    if not disaster_event:
        disaster_event = "unknown"

    disaster_type = (
        metadata.get("disaster_type")
        or metadata.get("disaster-type")
        or metadata.get("event_type")
        or label_data.get("disaster_type")
        or label_data.get("disaster-type")
        or label_data.get("event_type")
    )

    if disaster_type is None and disaster_event != "unknown":
        disaster_type = disaster_event.split("-")[0]

    disaster_type = normalize_disaster_type(disaster_type or "unknown")
    return disaster_event, disaster_type


def count_images(split_root):
    images_dir = split_root / "images_jpeg"
    pre_images = len(list(images_dir.glob("*_pre_disaster.*"))) if images_dir.exists() else 0
    post_images = len(list(images_dir.glob("*_post_disaster.*"))) if images_dir.exists() else 0
    return pre_images, post_images


def list_post_label_files(split_root):
    labels_dir = split_root / "labels"
    if not labels_dir.exists():
        raise FileNotFoundError(f"Missing labels directory: {labels_dir}")
    return sorted(labels_dir.glob("*_post_disaster.json"))


def summarize_split(split_name, split_root):
    label_files = list_post_label_files(split_root)
    pre_images, post_images = count_images(split_root)

    scene_damage_counter = Counter()
    building_damage_counter = Counter()
    disaster_type_counter = Counter()
    disaster_event_counter = Counter()
    scene_damage_by_disaster_type = Counter()

    total_buildings = 0

    for label_path in label_files:
        with label_path.open("r") as f:
            label_data = json.load(f)

        disaster_event, disaster_type = extract_disaster_fields(label_data)
        disaster_type_counter[disaster_type] += 1
        disaster_event_counter[disaster_event] += 1

        features = (label_data.get("features", {}) or {}).get("xy", []) or []
        scene_max_class = 0

        for feature in features:
            properties = feature.get("properties", {}) or {}
            if properties.get("feature_type") != "building":
                continue

            subtype = normalize_label_name(properties.get("subtype", "no-damage"))
            class_id = DAMAGE_NAME_TO_ID.get(subtype, 1)
            building_damage_counter[DAMAGE_ID_TO_NAME[class_id]] += 1
            scene_max_class = max(scene_max_class, class_id)
            total_buildings += 1

        scene_label = DAMAGE_ID_TO_NAME[scene_max_class]
        scene_damage_counter[scene_label] += 1
        scene_damage_by_disaster_type[(disaster_type, scene_label)] += 1

    summary_row = {
        "split": split_name,
        "pre_images": pre_images,
        "post_images": post_images,
        "post_label_files": len(label_files),
        "building_annotations": total_buildings,
    }

    return {
        "summary": summary_row,
        "scene_damage_counter": scene_damage_counter,
        "building_damage_counter": building_damage_counter,
        "disaster_type_counter": disaster_type_counter,
        "disaster_event_counter": disaster_event_counter,
        "scene_damage_by_disaster_type": scene_damage_by_disaster_type,
    }


def counter_rows(split_name, counter, label_col, preferred_order=None):
    total = sum(counter.values())
    labels = list(counter.keys())
    if preferred_order is not None:
        ordered_labels = [label for label in preferred_order if label in counter]
        ordered_labels += sorted(label for label in labels if label not in ordered_labels)
    else:
        ordered_labels = sorted(labels)

    rows = []
    for label in ordered_labels:
        count = counter.get(label, 0)
        pct = (count / total) if total else 0.0
        rows.append({"split": split_name, label_col: label, "count": count, "pct": pct})
    return rows


def plot_counts(df, label_col, title, figsize=(10, 5)):
    pivot = df.pivot(index=label_col, columns="split", values="count").fillna(0)
    ax = pivot.plot(kind="bar", figsize=figsize)
    ax.set_title(title)
    ax.set_xlabel(label_col.replace("_", " ").title())
    ax.set_ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
results = {split: summarize_split(split, DATA_ROOT / split) for split in SPLITS}

summary_df = pd.DataFrame([results[split]["summary"] for split in SPLITS]).set_index("split")
display(summary_df)

scene_rows = []
building_rows = []
type_rows = []
event_rows = []
cross_rows = []

for split in SPLITS:
    split_result = results[split]
    scene_rows.extend(counter_rows(split, split_result["scene_damage_counter"], "class_label", DAMAGE_ORDER))
    building_rows.extend(counter_rows(split, split_result["building_damage_counter"], "class_label", DAMAGE_ORDER[1:]))
    type_rows.extend(counter_rows(split, split_result["disaster_type_counter"], "disaster_type"))
    event_rows.extend(counter_rows(split, split_result["disaster_event_counter"], "disaster_event"))

    for (disaster_type, class_label), count in sorted(split_result["scene_damage_by_disaster_type"].items()):
        cross_rows.append(
            {
                "split": split,
                "disaster_type": disaster_type,
                "class_label": class_label,
                "count": count,
            }
        )

scene_class_df = pd.DataFrame(scene_rows)
building_class_df = pd.DataFrame(building_rows)
disaster_type_df = pd.DataFrame(type_rows)
disaster_event_df = pd.DataFrame(event_rows)
scene_damage_by_disaster_type_df = pd.DataFrame(cross_rows)

print("Scene-level damage counts")
display(scene_class_df.pivot(index="class_label", columns="split", values="count").fillna(0).astype(int))

print("Scene-level damage percentages")
display(scene_class_df.pivot(index="class_label", columns="split", values="pct").fillna(0).round(4))

print("Building-level damage counts")
display(building_class_df.pivot(index="class_label", columns="split", values="count").fillna(0).astype(int))

print("Building-level damage percentages")
display(building_class_df.pivot(index="class_label", columns="split", values="pct").fillna(0).round(4))

print("Disaster type counts")
display(disaster_type_df.pivot(index="disaster_type", columns="split", values="count").fillna(0).astype(int))

print("Disaster type percentages")
display(disaster_type_df.pivot(index="disaster_type", columns="split", values="pct").fillna(0).round(4))

print("Top disaster events per split")
for split in SPLITS:
    print(f"\n{split}")
    display(
        disaster_event_df[disaster_event_df["split"] == split]
        .sort_values(["count", "disaster_event"], ascending=[False, True])
        .head(15)
        .reset_index(drop=True)
    )

summary_df.to_csv(OUTPUT_DIR / "split_summary.csv")
scene_class_df.to_csv(OUTPUT_DIR / "scene_class_distribution.csv", index=False)
building_class_df.to_csv(OUTPUT_DIR / "building_class_distribution.csv", index=False)
disaster_type_df.to_csv(OUTPUT_DIR / "disaster_type_distribution.csv", index=False)
disaster_event_df.to_csv(OUTPUT_DIR / "disaster_event_distribution.csv", index=False)
scene_damage_by_disaster_type_df.to_csv(OUTPUT_DIR / "scene_damage_by_disaster_type.csv", index=False)

print(f"Saved CSV outputs to: {OUTPUT_DIR.resolve()}")


In [ ]:
plot_counts(scene_class_df, "class_label", "Scene-Level Damage Class Distribution")
plot_counts(building_class_df, "class_label", "Building-Level Damage Class Distribution")
plot_counts(disaster_type_df, "disaster_type", "Natural Disaster Type Distribution")
